# 04 - Feature Extraction and Transformation

This notebook turns the cleaned transfer dataset into machine-learning inputs.
It also documents the feature engineering decisions required by the project PDF.


## What this notebook does

1. Load the cleaned transfer dataset
2. Select leakage-safe input features
3. Split the data chronologically into train, validation, and test
4. Apply imputation, scaling, and one-hot encoding
5. Save the fitted preprocessing artifact and transformed matrices


In [9]:
from pathlib import Path
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.build_features import build_feature_splits, infer_feature_columns, save_preprocessor_artifact


In [10]:
DATA_DIR = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "models" / "transfer_success"

CLEAN_PATH = DATA_DIR / "transfer_modeling_dataset_clean.csv"
PREPROCESSOR_PATH = MODEL_DIR / "transfer_preprocessor.pkl"
FEATURE_PAYLOAD_PATH = MODEL_DIR / "feature_payload.joblib"

MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 1) Load the cleaned dataset


In [11]:
df = pd.read_csv(CLEAN_PATH, parse_dates=["transfer_date"])
print("Clean dataset shape:", df.shape)
df.head()


Clean dataset shape: (6675, 102)


,transfer_key,player_id,player_full_name,transfer_date,transfer_season,source_club_id,source_club_name,source_competition_id,source_competition_name,destination_club_id,...,target_is_eligible,transfer_fee_missing_flag,is_free_transfer,transfer_fee_for_model,transfer_fee_log1p,pre_transfer_market_value_log1p,market_value_in_eur_log1p,target_end_market_value_log1p,highest_market_value_in_eur_log1p,player_current_market_value_log1p
0,7825_2018-07-01_6195_5,7825,Pepe Reina,2018-07-01,18/19,6195,SSC Napoli,IT1,serie-a,5,...,True,0,1,0.0,0.000000,13.815512,13.815512,13.384729,16.906553,13.304687
1,11111_2018-07-01_29_1050,11111,Ramiro Funes Mori,2018-07-01,18/19,29,Everton,GB1,premier-league,1050,...,True,0,0,9000000.0,16.012735,16.118096,16.118096,15.384127,16.300417,11.225257
2,22141_2018-07-01_1025_12,22141,Antonio Mirante,2018-07-01,18/19,1025,Bologna,IT1,serie-a,12,...,True,0,0,4000000.0,15.201805,13.815512,13.815512,13.384729,15.607270,11.512935
3,27490_2018-07-01_1147_826,27490,Jean-Louis Leca,2018-07-01,18/19,1147,AC Ajaccio,FR1,ligue-1,826,...,True,0,0,300000.0,12.611541,13.815512,13.815512,13.592368,14.077876,12.206078
4,28140_2018-07-01_989_415,28140,Max Gradel,2018-07-01,18/19,989,Bournemouth,GB1,premier-league,415,...,True,0,0,2000000.0,14.508658,15.761421,15.761421,15.201805,16.012735,13.527830


## 2) Identify leakage-safe model inputs

We exclude:
- identifiers
- date fields
- any `target_*` columns
- the target itself


In [12]:
feature_columns = infer_feature_columns(df, target_col="transfer_success")

print("Selected feature count:", len(feature_columns))
pd.Series(feature_columns, name="feature_name").head(30)


Selected feature count: 57


0                  source_competition_id
1             destination_competition_id
2                        age_at_transfer
3                               position
4                           sub_position
5                                   foot
6                           height_in_cm
7                 country_of_citizenship
8       pre_transfer_market_value_source
9                market_value_180d_prior
10               market_value_365d_prior
11              market_value_change_180d
12              market_value_change_365d
13    transfer_fee_to_market_value_ratio
14       transfer_fee_minus_market_value
15               player_matches_180d_pre
16               player_minutes_180d_pre
17                 player_goals_180d_pre
18               player_assists_180d_pre
19           player_goals_per90_180d_pre
20         player_assists_per90_180d_pre
21               player_matches_365d_pre
22               player_minutes_365d_pre
23                 player_goals_365d_pre
24              

## 3) Chronological train / validation / test split

We use time order instead of a random split because transfer data is naturally time-based.
This is more realistic and safer against leakage.


In [13]:
payload = build_feature_splits(
    df,
    target_col="transfer_success",
    sort_column="transfer_date",
    train_ratio=0.70,
    val_ratio=0.15,
    feature_columns=feature_columns,
)

display(payload["split_summary"])


,split,rows,start_date,end_date
0,train,4672,2018-07-01,2021-06-30
1,val,1001,2021-06-30,2021-08-31
2,test,1002,2021-08-31,2022-06-30


## 4) Check class balance across the splits


In [14]:
split_distribution = pd.DataFrame(
    {
        "train": payload["y_train"].value_counts(normalize=True).sort_index(),
        "val": payload["y_val"].value_counts(normalize=True).sort_index(),
        "test": payload["y_test"].value_counts(normalize=True).sort_index(),
    }
).fillna(0)

display(split_distribution)

minority_rate = float(payload["y_train"].mean())
print("Training positive-class rate:", round(minority_rate, 4))
print("Balancing strategy: use class_weight in the models instead of synthetic oversampling.")


,train,val,test
transfer_success,,,
0,0.818707,0.795205,0.904192
1,0.181293,0.204795,0.095808


Training positive-class rate: 0.1813
Balancing strategy: use class_weight in the models instead of synthetic oversampling.


## 5) Inspect the transformed feature matrices

After preprocessing:
- numeric features use median imputation and standard scaling
- categorical features use most-frequent imputation and one-hot encoding


In [15]:
print("X_train shape:", payload["X_train"].shape)
print("X_val shape:", payload["X_val"].shape)
print("X_test shape:", payload["X_test"].shape)
print("Numeric columns:", len(payload["numeric_columns"]))
print("Categorical columns:", len(payload["categorical_columns"]))
print("Transformed features:", len(payload["feature_names"]))

payload["X_train"].head()


X_train shape: (4672, 220)
X_val shape: (1001, 220)
X_test shape: (1002, 220)
Numeric columns: 50
Categorical columns: 7
Transformed features: 220


,numeric__age_at_transfer,numeric__height_in_cm,numeric__market_value_180d_prior,numeric__market_value_365d_prior,numeric__market_value_change_180d,numeric__market_value_change_365d,numeric__transfer_fee_to_market_value_ratio,numeric__transfer_fee_minus_market_value,numeric__player_matches_180d_pre,numeric__player_minutes_180d_pre,...,categorical__country_of_citizenship_Uganda,categorical__country_of_citizenship_Ukraine,categorical__country_of_citizenship_United States,categorical__country_of_citizenship_Uruguay,categorical__country_of_citizenship_Uzbekistan,categorical__country_of_citizenship_Venezuela,categorical__country_of_citizenship_Wales,categorical__country_of_citizenship_Zimbabwe,categorical__pre_transfer_market_value_source_transfer_row_market_value,categorical__pre_transfer_market_value_source_valuation_history
0,3.489611,0.718156,-0.364168,-0.341774,-0.359718,-0.298777,-0.141440,0.240060,2.231198,2.868900,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,1.073612,0.571428,0.688471,0.779706,-0.629596,-0.473938,0.065876,0.240060,-0.222054,-0.487542,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3.187611,1.451795,-0.416800,-0.341774,-0.224779,-0.298777,0.779967,0.776033,1.653962,2.186619,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2.583612,-0.455667,-0.469431,-0.453922,-0.089841,-0.123615,-0.072335,0.280258,-0.799290,-0.713074,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,1.979612,-1.189306,0.162152,0.218966,-0.089841,-0.123615,-0.075625,-0.295913,1.653962,2.072905,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 6) Save reusable preprocessing artifacts


In [16]:
save_preprocessor_artifact(payload, PREPROCESSOR_PATH)

serializable_payload = {
    "X_train": payload["X_train"],
    "X_val": payload["X_val"],
    "X_test": payload["X_test"],
    "y_train": payload["y_train"],
    "y_val": payload["y_val"],
    "y_test": payload["y_test"],
    "feature_columns": payload["feature_columns"],
    "feature_names": payload["feature_names"],
    "numeric_columns": payload["numeric_columns"],
    "categorical_columns": payload["categorical_columns"],
    "split_summary": payload["split_summary"],
    "train_df": payload["train_df"],
    "val_df": payload["val_df"],
    "test_df": payload["test_df"],
}

joblib.dump(serializable_payload, FEATURE_PAYLOAD_PATH)

print("Saved preprocessor to:", PREPROCESSOR_PATH)
print("Saved feature payload to:", FEATURE_PAYLOAD_PATH)


Saved preprocessor to: D:\DataScience\project\ScoutRadar\models\transfer_success\transfer_preprocessor.pkl
Saved feature payload to: D:\DataScience\project\ScoutRadar\models\transfer_success\feature_payload.joblib
